# Q1 · Students: baselines + knowledge distillation
Expected on a T4: 20 epochs each: baselines ~20–30 min, KD ~30–40 min (all three students train together).

### Setup (Runtime → Change runtime type → **T4 GPU**)
Before running, drag these files into the Colab **Files** pane (they land in `/content`):
- `cifar100.tar.gz`
- `teacher_outputs.tar.gz` (downloaded from `colab_teacher.ipynb`)
If a file is missing, the cell opens a file picker instead.

No Google Drive: all outputs live in `/content/out` (same layout as the repo) and are
**lost when the runtime disconnects**. Keep the tab active and run the download cells as soon as
a stage finishes.

In [ ]:
# Private repo? use: https://<GITHUB_TOKEN>@github.com/guzorpazorp/edge_compute_assignment_4.git
REPO = 'https://github.com/guzorpazorp/edge_compute_assignment_4.git'
!test -d /content/repo || git clone -q $REPO /content/repo
%cd /content/repo
!git pull -q
!pip install -q optuna onnx onnxruntime onnxscript

In [ ]:
import os, subprocess, tarfile
from google.colab import files

OUT = '/content/out'                        # mirrors the repo: out/checkpoints, out/results
os.environ['CKPT_DIR'] = f'{OUT}/checkpoints'
os.environ['RESULTS_DIR'] = f'{OUT}/results'
os.environ['DATA_DIR'] = '/content/data'

def get_upload(name):
    """Path of a manually uploaded file: Files-pane drop in /content, else a file picker."""
    path = f'/content/{name}'
    if not os.path.exists(path):
        print(f'{name} not found in /content, choose it in the picker')
        uploaded = files.upload()               # saves into the current dir
        assert name in uploaded, f'expected {name}, got {list(uploaded)}'
        os.replace(name, path)
    return path

def extract(name, dest):
    with tarfile.open(get_upload(name)) as t:
        t.extractall(dest, filter='data')

def download_outputs(name, patterns):
    """Bundle outputs (paths relative to OUT) -> extract locally with: tar -xzf NAME -C <repo>"""
    subprocess.run(f"cd {OUT} && tar -czf /content/{name} {' '.join(patterns)}", shell=True, check=True)
    files.download(f'/content/{name}')

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Dataset: manual upload of cifar100.tar.gz (made locally from data/cifar-100-python)
if not os.path.exists('/content/data/cifar-100-python/train'):
    extract('cifar100.tar.gz', '/content/data')
!python scripts/download_data.py --dataset cifar100      # only verifies, files are already present

In [ ]:
# Teacher checkpoint (needed for KD)
if not os.path.exists(f'{OUT}/checkpoints/teacher_resnet50/best.pt'):
    extract('teacher_outputs.tar.gz', OUT)
!ls -la {OUT}/checkpoints/teacher_resnet50

## Baselines (hard labels only)

In [ ]:
!python scripts/train_student.py --mode baseline

In [ ]:
download_outputs('baseline_outputs.tar.gz', ['checkpoints/baseline_*/best.pt', 'results/logs/baseline_*.csv'])

## Knowledge distillation (soft targets + T + hard labels + feature KD)

In [ ]:
!python scripts/train_student.py --mode kd

In [ ]:
download_outputs('kd_outputs.tar.gz', ['checkpoints/kd_*/best.pt', 'results/logs/kd_*.csv'])

In [ ]:
import pandas as pd
rows = []
for mode in ('baseline', 'kd'):
    for s in ('mbv2', 'squeezenet', 'efflite0'):
        p = f'{OUT}/results/logs/{mode}_{s}.csv'
        if os.path.exists(p):
            rows.append({'run': f'{mode}_{s}', 'best_top1': pd.read_csv(p).test_acc.max()})
pd.DataFrame(rows)